# Visualization Best Practices

## Introduction

A chart is a compressed argument. Every element — axes, scale, color, annotations — either helps or hinders the reader's understanding. This notebook covers the principles of honest, effective data visualization: what makes a chart work, how charts mislead, and how to fix common problems.

## Objectives

You will be able to:

* Apply visual encoding principles to choose appropriate chart types and encodings
* Recognize and fix misleading charts (truncated axes, dual axes, pie charts)
* Use pre-attentive attributes effectively: color, position, size
* Design for accessibility: colorblind-safe palettes, labels, annotations
* Iterate from an exploratory draft to a presentation-ready figure

---

## Visual Encoding Hierarchy

Cleveland and McGill (1984) found that humans decode visual encodings with different accuracy. Listed from most to least accurate:

1. **Position along a common scale** — bar chart height, scatter plot position
2. **Position on non-aligned scales** — small multiples
3. **Length** — bar charts
4. **Angle / slope** — line chart slope, pie slices
5. **Area** — bubble charts, treemaps
6. **Volume / density** — 3D charts, density maps
7. **Color hue** — least accurate for quantitative comparisons

> Use the most accurate encoding for the most important comparison. Reserve color and size for secondary dimensions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})

---

## Misleading Chart #1 — Truncated Y-Axis

In [ ]:
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
values = [982, 985, 986, 990, 992, 994]  # a 1.2% change

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: truncated axis — makes 1.2% change look enormous
axes[0].bar(months, values, color='steelblue')
axes[0].set_ylim(978, 998)
axes[0].set_title('MISLEADING: Truncated Axis')
axes[0].set_ylabel('Sales Index')

# Right: starts at zero — honest representation
axes[1].bar(months, values, color='steelblue')
axes[1].set_ylim(0, 1100)
axes[1].set_title('HONEST: Zero Baseline')
axes[1].set_ylabel('Sales Index')

plt.suptitle('The same data — two very different stories', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Actual change: {(values[-1]-values[0])/values[0]*100:.1f}%")
print("Rule: Bar charts must start at zero. Line charts may use a narrow range — but label clearly.")

---

## Misleading Chart #2 — The Problematic Pie Chart

In [ ]:
categories = ['Engineering', 'Marketing', 'Sales', 'DS', 'Support']
sizes      = [34, 28, 22, 18, 16]  # similar-ish values — hard to rank in a pie

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: pie chart — hard to rank slices accurately
axes[0].pie(sizes, labels=categories, autopct='%1.0f%%', startangle=90)
axes[0].set_title('WORSE: Pie Chart\n(hard to rank slices)')

# Right: horizontal bar — easy to rank, compare
sorted_sizes = sorted(zip(sizes, categories))
axes[1].barh([c for s, c in sorted_sizes], [s for s, c in sorted_sizes], color='steelblue')
axes[1].set_title('BETTER: Sorted Bar Chart')
axes[1].set_xlabel('Headcount')

plt.tight_layout()
plt.show()

print("Use pie only when parts-of-a-whole AND there are ≤ 3 categories AND they differ substantially.")

---

## Misleading Chart #3 — Dual Y-Axes

In [ ]:
# Dual axes let you make any two unrelated trends look correlated
# by scaling one axis to fit the other

x = range(10)
trend_a = [10, 15, 12, 18, 22, 25, 20, 30, 28, 35]  # sales
trend_b = [100, 95, 90, 85, 95, 110, 100, 120, 115, 130]  # unrelated: temperature

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.plot(x, trend_a, 'b-o', label='Sales (left)')
ax2.plot(x, trend_b, 'r-s', label='Temperature (right)')
ax1.set_ylabel('Sales', color='b')
ax2.set_ylabel('Temperature °F', color='r')
ax1.set_title('MISLEADING: Dual Axes — two different scales suggest false correlation')
plt.tight_layout()
plt.show()

print("Fix: use two separate charts with shared x-axis, or normalize both series to [0,1].")

---

## Pre-Attentive Attributes

In [ ]:
# Pre-attentive attributes are processed by the brain before conscious attention
# Use them to guide the reader's eye to the key finding

depts = ['Engineering', 'Marketing', 'Data Science', 'Sales', 'Support']
salaries = [92000, 75000, 108000, 68000, 58000]

# Highlight the standout finding with color
colors = ['#2196F3' if d == 'Data Science' else '#BBDEFB' for d in depts]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(depts, salaries, color=colors)
ax.set_xlabel('Average Salary ($)')
ax.set_title('Data Science commands the highest salaries', fontsize=13, fontweight='bold')

# Annotation draws eye directly to the insight
ax.annotate('$108k', xy=(108000, 2), fontsize=12, color='#2196F3',
            fontweight='bold', va='center', ha='right')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---

## Accessibility — Colorblind-Safe Design

In [ ]:
# ~8% of men have red-green color blindness
# Never use red/green alone to encode meaning

# Good colorblind-safe palettes:
cb_palettes = {
    'Okabe-Ito (qualitative)': ['#000000', '#E69F00', '#56B4E9', '#009E73',
                                  '#F0E442', '#0072B2', '#D55E00', '#CC79A7'],
    'Viridis (sequential)':    'Use plt.cm.viridis',
    'Colorbrewer diverging':   'Use plt.cm.RdBu (diverging, colorblind safe)',
}

for name, colors in cb_palettes.items():
    print(f"{name}: {colors}")

# Demo: Okabe-Ito palette
okabe = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2']
fig, ax = plt.subplots(figsize=(8, 1.5))
for i, c in enumerate(okabe):
    ax.add_patch(plt.Rectangle((i, 0), 0.9, 1, color=c))
    ax.text(i + 0.45, 0.5, c, ha='center', va='center', fontsize=9)
ax.set_xlim(0, 5)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Okabe-Ito Colorblind-Safe Palette')
plt.tight_layout()
plt.show()

---

## Chart Anatomy — Getting Polish Right

In [ ]:
# Draft → polished example
np.random.seed(0)
x = np.linspace(0, 10, 50)
y = 2 * x + np.random.normal(0, 3, 50)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Draft
axes[0].scatter(x, y)
axes[0].set_title('Draft')

# Polished
axes[1].scatter(x, y, alpha=0.65, edgecolor='white', s=60, color='#0072B2', linewidth=0.5)
m, b = np.polyfit(x, y, 1)
axes[1].plot(x, m*x + b, 'r--', linewidth=1.5, label=f'Trend: y = {m:.1f}x + {b:.1f}')
axes[1].set_title('Experience and Salary — Positive Relationship', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Years of Experience')
axes[1].set_ylabel('Salary ($000s)')
axes[1].legend()
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---

## Best Practices Checklist

**Title:** State the finding, not just the topic. "DS earns more" not "Salary by Department".

**Axes:** Always label both axes with units. Start bars at zero.

**Color:** Use sparingly — color one thing if it's the key comparison. Use colorblind-safe palettes.

**Annotations:** Add direct labels or callouts instead of relying on legends when possible.

**Ink ratio:** Remove gridlines, borders, and ticks that add ink without adding information.

**Context:** Add a data source note for external presentation.

---

## Practice

In [ ]:
# Critique and fix the following chart
categories = ['A', 'B', 'C', 'D', 'E']
values     = [52, 49, 50, 51, 53]

# Original (problematic)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(categories, values, color=['red', 'green', 'blue', 'orange', 'purple'])
ax.set_ylim(48, 54)
ax.set_title('Category Comparison')
plt.show()

# TODO: Fix the issues:
# 1. Truncated axis — start at 0 (or use a line/dot if narrow range is genuinely needed)
# 2. Rainbow colors for non-categorical dimension — use one color
# 3. Vague title — write a title that states the finding

## Summary

| Principle | Rule |
|-----------|------|
| Zero baseline | Bar charts always start at 0 |
| Encoding | Use position > length > area > color for key comparisons |
| Color | Colorblind-safe palettes; one highlighted color for the key finding |
| Pie charts | Only when ≤ 3 very different slices; otherwise use bars |
| Dual axes | Avoid — they create false correlations |
| Title | State the insight, not the topic |
| Ink | Remove decorative elements that don't encode data |

Next: Matplotlib — fine-grained control over figures, subplots, and custom styling.